# Step 06: Baseline Model — Logistic Regression

## Overview
This notebook builds an interpretable Logistic Regression baseline for attrition prediction:
- Stratified 80/20 train/test split to preserve positive class ratio (16.1%).
- Scikit-Learn `ColumnTransformer` with `OneHotEncoder` and `StandardScaler`.
- Cost-aware modeling (`class_weight='balanced'`).
- Primary evaluation on **Recall**, **F1-Score**, and **ROC-AUC** (Ground Rule 4).


In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix

PROCESSED_DIR = os.path.join("..", "data", "processed")
df = pd.read_csv(os.path.join(PROCESSED_DIR, "features_engineered.csv"))

# Drop non-predictive identifier and raw target strings
drop_cols = ['EmployeeNumber', 'Employee ID', 'Attrition', 'Target_Attrition']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols]
y = df['Target_Attrition']

print(f"Features shape: {X.shape}, Target shape: {y.shape}")
print(f"Target distribution:\n{y.value_counts(normalize=True)}")


---
## 1. Stratified Train-Test Split & Preprocessing Pipeline


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
    ]
)

baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
])

baseline_pipeline.fit(X_train, y_train)
print("✔ Baseline Logistic Regression Trained Successfully.")


---
## 2. Baseline Model Evaluation (Recall & ROC-AUC Priority)


In [ ]:
y_pred = baseline_pipeline.predict(X_test)
y_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred)

print("=== Baseline Model Metrics (Logistic Regression) ===")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}  <-- Priority Metric")
print(f"F1-Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print("\nConfusion Matrix:")
print(pd.DataFrame(cm, index=['Actual Stay (0)', 'Actual Leave (1)'], columns=['Pred Stay (0)', 'Pred Leave (1)']))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Stay (0)', 'Leave (1)']))
